# ⚡ 100% 合規雲端 GPU 算圖與算影片極限測試工作站 (`auto_worker.ipynb`)

- **完全符合 Kaggle & Google Colab 規範**：不開啟任何第三方反向代理通道（無 `cloudflared`），完全透過官方 API 執行並取回產出檔案。
- **多樣化隨機時尚寫真 + 極限壓測**：鎖定「20歲法越混血、白皙透亮瓷肌、165cm 50kg 沙漏體態」核心人設，自動隨機抽樣不重複的臉型、髮色、髮型、妝容、時尚蕾絲連身衣款式/顏色、姿勢與場景，並測試標準高畫質 (`832x1216`) 與超高解析度 (`1024x1536`) 的算圖極限與 VRAM 峰值！
- **照片轉動態影片 (I2V)**：將兩張參考圖 (`ref_1.jpg`, `ref_2.jpg`) 轉為 5 秒 (`81` 影格) 自然動態 MP4 影片，並記錄完整硬體效能數據。

In [ ]:
import os, sys, json, time, shutil, subprocess, gc

print("🚀 [1/5] 正在安裝最新版 Diffusers 與加速套件...")
subprocess.run("pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg ftfy", shell=True, check=True)

print("🔄 [2/5] 正在從 GitHub 同步最新任務設定與隨機 Prompt 引擎...")
if not os.path.exists("colab-ai-media-lab"):
    subprocess.run("git clone https://github.com/Chenponpon/colab-ai-media-lab.git", shell=True, check=True)
else:
    subprocess.run("cd colab-ai-media-lab && git fetch --all && git reset --hard origin/main", shell=True, check=True)

sys.path.insert(0, os.path.abspath("colab-ai-media-lab/scripts"))
from prompt_randomizer import generate_diversified_prompt

with open("colab-ai-media-lab/queue/current_job.json", "r", encoding="utf-8") as f:
    job = json.load(f)

import torch, psutil
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
gpu_name = f"{torch.cuda.get_device_name(0)} (x{gpu_count})" if gpu_count > 0 else "CPU"
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if gpu_count > 0 else 0
ram_gb = round(psutil.virtual_memory().total / (1024**3), 2)
print(f"✅ 硬體檢測完成 | GPU: {gpu_name} | 單卡 VRAM: {vram_gb} GB | System RAM: {ram_gb} GB")

from PIL import Image
image_times = []
image_peak_vram = 0.0
max_res_str = "832x1216"

# ==================== 階段一：多樣化隨機生圖 + 超高解析度極限壓測 ====================
img_cfg = job.get("image_generation", {})
if img_cfg.get("enabled", True):
    print("🎨 [3/5] 正在載入 RealVisXL V5.0 寫實人像模型並執行生圖極限壓測...")
    from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler
    model_id = img_cfg.get("model_id", "SG161222/RealVisXL_V5.0")
    pipe = AutoPipelineForText2Image.from_pretrained(model_id, torch_dtype=torch.float16, variant="fp16").to("cuda")
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True, algorithm_type="sde-dpmsolver++")
    pipe.vae.enable_tiling()

    num_imgs = img_cfg.get("num_images", 4)
    for i in range(num_imgs):
        # 前 3 張使用標準直式高畫質 832x1216，第 4 張挑戰超高解析度 1024x1536 測試算圖極限！
        w, h = (1024, 1536) if i == num_imgs - 1 else (img_cfg.get("width", 832), img_cfg.get("height", 1216))
        active_prompt, recipe = generate_diversified_prompt()

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        gen = torch.Generator("cuda").manual_seed(recipe["seed"])
        img = pipe(
            prompt=active_prompt,
            negative_prompt=img_cfg.get("negative_prompt", ""),
            width=w, height=h,
            num_inference_steps=img_cfg.get("steps", 28),
            guidance_scale=img_cfg.get("guidance_scale", 5.5),
            generator=gen
        ).images[0]
        dt = round(time.time() - t0, 2)
        pk = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
        image_times.append(dt)
        image_peak_vram = max(image_peak_vram, pk)
        max_res_str = f"{w}x{h}"

        recipe["elapsed_sec"] = dt
        recipe["peak_vram_gb"] = pk
        recipe["resolution"] = f"{w}x{h}"

        fname = f"generated_portrait_{i+1}.jpg"
        rname = f"generated_portrait_{i+1}_recipe.json"
        img.save(fname, quality=95)
        with open(rname, "w", encoding="utf-8") as rf:
            json.dump(recipe, rf, ensure_ascii=False, indent=2)

        print(f"\n================ 🎲 第 {i+1} 張 [{w}x{h}] 耗時: {dt}s | 峰值 VRAM: {pk} GB ================")
        print(recipe["summary_text"])

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

# ==================== 階段二：照片轉動態影片 (I2V) 極限壓測 ====================
vid_cfg = job.get("video_generation", {})
video_times = []
video_peak_vram = 0.0
video_model_used = "Lightricks/LTX-Video"

if vid_cfg.get("enabled", True):
    print("\n🎬 [4/5] 正在載入圖生影引擎 (LTX-Video I2V) 將你的參考圖轉為 5 秒動態影片...")
    from diffusers import LTXImageToVideoPipeline
    from diffusers.utils import export_to_video

    v_pipe = LTXImageToVideoPipeline.from_pretrained("Lightricks/LTX-Video", torch_dtype=torch.bfloat16)
    v_pipe.enable_model_cpu_offload()
    v_pipe.vae.enable_tiling()

    targets = [os.path.join("colab-ai-media-lab", p) for p in vid_cfg.get("input_images", [])]
    vw, vh = vid_cfg.get("width", 512), vid_cfg.get("height", 768)
    for idx, img_p in enumerate(targets):
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        in_img = Image.open(img_p).convert("RGB").resize((vw, vh), Image.LANCZOS)
        frames = v_pipe(
            image=in_img,
            prompt=vid_cfg["motion_prompt"],
            negative_prompt=vid_cfg.get("negative_prompt", ""),
            height=vh, width=vw,
            num_frames=vid_cfg.get("num_frames", 81),
            guidance_scale=vid_cfg.get("guidance_scale", 5.0),
            num_inference_steps=vid_cfg.get("steps", 25)
        ).frames[0]
        dt = round(time.time() - t0, 2)
        pk = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
        video_times.append(dt)
        video_peak_vram = max(video_peak_vram, pk)

        vname = f"animated_video_{idx+1}.mp4"
        export_to_video(frames, vname, fps=vid_cfg.get("fps", 16))
        print(f"✅ 影片完成: {vname} ({vw}x{vh}, 81影格/5秒) | 耗時: {dt}s | 峰值 VRAM: {pk} GB")

# ==================== 階段三：輸出硬體壓測總結報告 ====================
bench_report = {
    "gpu_name": gpu_name,
    "vram_gb": vram_gb,
    "ram_gb": ram_gb,
    "avg_image_time_sec": round(sum(image_times)/len(image_times), 2) if image_times else 0,
    "image_peak_vram_gb": image_peak_vram,
    "max_resolution_tested": max_res_str,
    "avg_video_time_sec": round(sum(video_times)/len(video_times), 2) if video_times else 0,
    "video_peak_vram_gb": video_peak_vram,
    "video_model": video_model_used
}
with open("benchmark_report.json", "w", encoding="utf-8") as bf:
    json.dump(bench_report, bf, ensure_ascii=False, indent=2)

print("\n🏆 [5/5] 雲端極限壓測與產出全部完成！報告已存為 benchmark_report.json。")